# 01 - Chain Snapshot

Load the full option chain for one underlying at the latest snapshot.

Shows:
- every strike and expiry currently active
- mark price, bid/ask, IV, OI, volume
- underlying spot at snapshot time


## Setup

In [1]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

options.db: D:\Projects\done\eth_bot\eth_bot_v2\database\options.db
market.db : D:\Projects\done\eth_bot\eth_bot_v2\database\market.db


## 1. Latest snapshot

In [2]:
oc = opt("""
    SELECT * FROM option_chain
    WHERE underlying='ETH'
      AND ts = (SELECT MAX(ts) FROM option_chain WHERE underlying='ETH')
    ORDER BY expiry, strike, kind
""")
print("rows:", len(oc))
if not oc.empty:
    print("snapshot:", oc.date.iloc[0])
    print("expiries:", oc.expiry.nunique())
    print("strikes :", oc.strike.nunique())
oc.head(20)

rows: 806
snapshot: 2026-10-08 12:36:37
expiries: 12
strikes : 89


,underlying,expiry,strike,kind,ts,mark_price,bid,ask,bid_iv,ask_iv,mark_iv,oi,volume,underlying_px,date
0,ETH,1791504000000,2000.0,C,1791462997815,0.209025,0.2080,0.2115,None,None,1.5485,0.0,0.0,2528.64,2026-10-08 12:36:37
1,ETH,1791504000000,2000.0,P,1791462997815,0.000011,0.0000,0.0001,None,None,1.5485,3776.0,336.0,2528.64,2026-10-08 12:36:37
2,ETH,1791504000000,2100.0,C,1791462997815,0.169564,0.1685,0.1725,None,None,1.5249,0.0,0.0,2528.64,2026-10-08 12:36:37
3,ETH,1791504000000,2100.0,P,1791462997815,0.000099,0.0000,0.0001,None,None,1.5249,2194.0,1361.0,2528.72,2026-10-08 12:36:37
4,ETH,1791504000000,2200.0,C,1791462997815,0.130156,0.1290,0.1325,None,None,1.3124,0.0,0.0,2528.64,2026-10-08 12:36:37
5,ETH,1791504000000,2200.0,P,1791462997815,0.000241,0.0002,0.0003,None,None,1.3124,6325.0,2423.0,2528.64,2026-10-08 12:36:37
6,ETH,1791504000000,2300.0,C,1791462997815,0.090892,0.0900,0.0935,None,None,1.0598,54.0,18.0,2528.64,2026-10-08 12:36:37
7,ETH,1791504000000,2300.0,P,1791462997815,0.000526,0.0005,0.0006,None,None,1.0598,3610.0,1053.0,2528.72,2026-10-08 12:36:37
8,ETH,1791504000000,2350.0,C,1791462997815,0.071454,0.0705,0.0740,None,None,0.9388,0.0,0.0,2528.64,2026-10-08 12:36:37
9,ETH,1791504000000,2350.0,P,1791462997815,0.000862,0.0007,0.0010,None,None,0.9388,2901.0,546.0,2528.64,2026-10-08 12:36:37


## 2. Expiry summary

In [3]:
if not oc.empty:
    oc["expiry_date"] = pd.to_datetime(oc.expiry, unit="ms").dt.date
    summary = oc.groupby("expiry_date").agg(
        instruments=("strike", "count"),
        strikes=("strike", "nunique"),
        total_oi=("oi", "sum"),
        total_vol=("volume", "sum"),
    )
    summary

## 3. Spot price

In [4]:
if not oc.empty:
    print("underlying spot:", oc.underlying_px.iloc[0])

underlying spot: 2528.64
